# File Organizer
### Progetto Python | Organizzazione automatica di file ed analisi immagini con NumPy

**Autore:** Valerio Aquilani
**Corso:** Python — start2impact University

**Contesto:** partendo da una cartella `files` contenente 8 file eterogenei (2 di testo, 2 audio, 4 immagini), ho sviluppato uno script che organizza automaticamente il contenuto in sottocartelle per tipo, tiene traccia di ogni spostamento in un log CSV, e analizza le immagini con NumPy per ricavarne i valori medi dei canali colore.

## Step 1 — Organizzazione automatica

Ho scritto uno script Python che itera in ordine alfabetico sui file della cartella `files` e, in base al tipo (audio, documento, immagine), li sposta nella sottocartella corrispondente, creandola automaticamente se non esiste. Durante il ciclo, lo script stampa nome, tipo e dimensione in byte di ogni file spostato:

In [1]:
# Import delle librerie necessarie
import os
import shutil
import csv

CARTELLA_FILES = "files"
RECAP_PATH = os.path.join(CARTELLA_FILES, "recap.csv")

# Mappa estensione -> (tipo, nome della sottocartella di destinazione)
MAPPA_TIPI = {
    ".mp3": ("audio", "audio"),
    ".txt": ("doc", "docs"),
    ".odt": ("doc", "docs"),
    ".jpg": ("image", "images"),
    ".jpeg": ("image", "images"),
    ".png": ("image", "images"),
}

def get_tipo_e_sottocartella(nome_file):
    """Restituisce la tupla (tipo, sottocartella) corrispondente all'estensione del file."""
    _, estensione = os.path.splitext(nome_file)
    return MAPPA_TIPI.get(estensione.lower(), ("other", "others"))

def aggiungi_a_recap(righe):
    """Aggiunge righe al file recap.csv. Se il file non esiste ancora, lo crea con
    l'intestazione. Le righe vengono sempre AGGIUNTE (append), mai sovrascritte,
    cosi' che esecuzioni successive dello script aggiornino il recap senza perdere
    lo storico dei file gia' spostati in precedenza."""
    file_esiste = os.path.isfile(RECAP_PATH)
    with open(RECAP_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_esiste:
            writer.writerow(["name", "type", "size(B)"])
        writer.writerows(righe)

# Elenco alfabetico dei soli file presenti direttamente dentro files/
# (escludo le sottocartelle gia' create e il file recap.csv stesso)
nomi_file = sorted(
    f for f in os.listdir(CARTELLA_FILES)
    if os.path.isfile(os.path.join(CARTELLA_FILES, f)) and f != "recap.csv"
)

righe_recap = []

for nome_file in nomi_file:
    percorso_sorgente = os.path.join(CARTELLA_FILES, nome_file)
    nome_senza_estensione, _ = os.path.splitext(nome_file)
    dimensione = os.path.getsize(percorso_sorgente)
    tipo, sottocartella = get_tipo_e_sottocartella(nome_file)

    # Stampo nome (senza estensione), tipo e dimensione in byte
    print(f"{nome_senza_estensione} type:{tipo} size:{dimensione}B")

    # Creo la sottocartella di destinazione se non esiste ancora
    cartella_destinazione = os.path.join(CARTELLA_FILES, sottocartella)
    os.makedirs(cartella_destinazione, exist_ok=True)

    # Sposto il file nella sottocartella di competenza
    shutil.move(percorso_sorgente, os.path.join(cartella_destinazione, nome_file))

    righe_recap.append([nome_senza_estensione, tipo, dimensione])

# Aggiorno il recap con i file appena spostati, senza toccare le righe precedenti
if righe_recap:
    aggiungi_a_recap(righe_recap)


bw type:image size:94926B
ciao type:doc size:12B
daffodil type:image size:24657B
eclipse type:image size:64243B
pippo type:doc size:8299B
song1 type:audio size:1087849B
song2 type:audio size:764176B
trump type:image size:10195B


Oltre alla stampa a video, ogni spostamento viene registrato in un file `recap.csv`, con le stesse informazioni (nome, tipo, dimensione). Il file viene sempre aggiornato in append, mai sovrascritto: se lo script viene rilanciato più volte, il recap accumula lo storico di tutti i file spostati senza perdere le righe precedenti.

La struttura finale della cartella `files` è:

```
files
├── audio
│   ├── song1.mp3
│   └── song2.mp3
├── docs
│   ├── ciao.txt
│   └── pippo.odt
├── images
│   ├── bw.png
│   ├── daffodil.jpg
│   ├── eclipse.png
│   └── trump.jpeg
└── recap.csv
```

## Step 2 — Interfaccia a linea di comando (CLI)

Ho estratto la logica dello Step 1 in un eseguibile standalone, `addfile.py` (visibile in fondo a questa pagina), dotato di interfaccia a linea di comando tramite `argparse`. Lo script sposta un singolo file passato come argomento nella sua sottocartella di competenza, aggiornando il recap. Se il file indicato non esiste, lo comunica all'utente invece di generare un errore.

## Step 3 — Analisi dei canali colore con NumPy

Un'immagine in scala di grigio ha un solo livello di colore, una RGB ne ha 3, una RGBA 4 (il quarto è il canale *alpha*, la trasparenza). Ho usato il modulo `Image` di PIL per caricare ogni immagine della sottocartella `images`, convertendola in un array NumPy con `np.array`: dalla forma dell'array (`shape`) si ricava se l'immagine è in scala di grigio, RGB o RGBA, e con `.mean()` calcolo il valore medio di ciascun canale.

Il risultato è una tabella riassuntiva (costruita con la libreria `tabulate`) con dimensioni e valori medi dei canali per ogni immagine:

In [2]:
# Import delle librerie necessarie per lo Step 3
import numpy as np
from PIL import Image
from tabulate import tabulate

CARTELLA_IMMAGINI = os.path.join(CARTELLA_FILES, "images")

righe_tabella = []

for nome_file in sorted(os.listdir(CARTELLA_IMMAGINI)):
    percorso = os.path.join(CARTELLA_IMMAGINI, nome_file)
    if not os.path.isfile(percorso):
        continue

    nome_senza_estensione, _ = os.path.splitext(nome_file)
    arr = np.array(Image.open(percorso))

    altezza, larghezza = arr.shape[0], arr.shape[1]
    grayscale = r = g = b = alpha = 0.0

    if arr.ndim == 2:
        # Immagine in scala di grigio: un solo livello di colore.
        # La media si calcola sull'intero array, senza bisogno di axis.
        grayscale = arr.mean()
    else:
        # Immagine RGB (3 canali) o RGBA (4 canali): calcolo la media
        # di ciascun canale separatamente, indicizzando l'ultimo asse.
        n_canali = arr.shape[2]
        r = arr[:, :, 0].mean()
        g = arr[:, :, 1].mean()
        b = arr[:, :, 2].mean()
        if n_canali == 4:
            alpha = arr[:, :, 3].mean()

    righe_tabella.append([
        nome_senza_estensione, altezza, larghezza,
        round(grayscale, 2), round(r, 2), round(g, 2), round(b, 2), round(alpha, 2)
    ])

intestazione = ["name", "height", "width", "grayscale", "R", "G", "B", "ALPHA"]
print(tabulate(righe_tabella, headers=intestazione, tablefmt="fancy_grid", floatfmt=".2f"))


╒══════════╤══════════╤═════════╤═════════════╤════════╤════════╤═══════╤═════════╕
│ name     │   height │   width │   grayscale │      R │      G │     B │   ALPHA │
╞══════════╪══════════╪═════════╪═════════════╪════════╪════════╪═══════╪═════════╡
│ bw       │      512 │     512 │       21.48 │   0.00 │   0.00 │  0.00 │    0.00 │
├──────────┼──────────┼─────────┼─────────────┼────────┼────────┼───────┼─────────┤
│ daffodil │      500 │     335 │        0.00 │ 109.23 │  85.52 │  4.77 │    0.00 │
├──────────┼──────────┼─────────┼─────────────┼────────┼────────┼───────┼─────────┤
│ eclipse  │      256 │     256 │        0.00 │ 109.05 │ 109.52 │ 39.85 │  133.59 │
├──────────┼──────────┼─────────┼─────────────┼────────┼────────┼───────┼─────────┤
│ trump    │      183 │     275 │        0.00 │  97.01 │  98.99 │ 90.92 │    0.00 │
╘══════════╧══════════╧═════════╧═════════════╧════════╧════════╧═══════╧═════════╛


Oltre al nome del file, la tabella riporta:

- altezza e larghezza dell'immagine, in pixel
- per le immagini in scala di grigio, la colonna *grayscale* con la media dell'unico livello di colore
- per le immagini a colori, la media di ciascun canale (R, G, B, e ALPHA se presente)